In [ ]:
# CELL 0: SMART CHECK + FETCH FROM GITHUB RAW URL
import os, json, hashlib, requests, time, sys

print("🔍 Checking for new pipeline data...")

# GitHub Raw URL (public, no auth needed)
PIPELINE_URL = "https://raw.githubusercontent.com/muhammadasjad2008/content-factory-orchestrator/main/pipeline_data.json"
LEDGER_PATH = "/kaggle/working/.pipeline_hash.json"

try:
    # Fetch current pipeline data
    resp = requests.get(PIPELINE_URL, timeout=30)
    resp.raise_for_status()
    pipeline = resp.json()
    current_hash = hashlib.md5(resp.content).hexdigest()
except Exception as e:
    print(f"⚠️ Could not fetch pipeline data: {e}. Exiting.")
    sys.exit(0)

# Check local hash ledger
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH, 'r') as f:
        last_hash = json.load(f).get('hash')
    if last_hash == current_hash:
        print(f"✅ No new data (hash unchanged). Exiting in 5s to save GPU time.")
        time.sleep(5)
        sys.exit(0)  # Clean exit - no error

# Save new hash for next run
os.makedirs(os.path.dirname(LEDGER_PATH), exist_ok=True)
with open(LEDGER_PATH, 'w') as f:
    json.dump({'hash': current_hash, 'checked_at': time.time()}, f)

print(f"🎯 New data detected! Proceeding with pipeline...")

# Extract pipeline data
reel_url = pipeline.get('reel_url', '')
shortcode = pipeline.get('shortcode', '')
username = pipeline.get('username', 'unknown')
script_text = pipeline.get('script_text', '')

if not shortcode:
    print("❌ No shortcode in pipeline_data.json. Exiting.")
    sys.exit(0)

print(f"🎯 Target: {reel_url} | Shortcode: {shortcode}")

# CELL 1: INSTALL DEPENDENCIES (if not already installed)
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null
!pip install -q requests torch transformers scipy accelerate google-api-python-client google-auth-oauthlib google-auth-httplib2 instaloader
print('✅ Dependencies ready')

# CELL 2: IMPORTS & CONFIG
import os, json, re, requests, subprocess, time, random, torch
from kaggle_secrets import UserSecretsClient
from transformers import AutoProcessor, BarkModel
import scipy.io.wavfile as wavfile
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
import instaloader

print('🔐 Loading secrets...')
secrets = UserSecretsClient()
GH_TOKEN = secrets.get_secret('GH_TOKEN')
YT_CLIENT_ID = secrets.get_secret('YOUTUBE_CLIENT_ID')
YT_CLIENT_SECRET = secrets.get_secret('YOUTUBE_CLIENT_SECRET')
YT_REFRESH_TOKEN = secrets.get_secret('YOUTUBE_REFRESH_TOKEN')

GITHUB_USER = 'muhammadasjad2008'
GITHUB_REPO = 'content-factory-orchestrator'
BRANCH = 'main'

WORKING_DIR = '/kaggle/working'
RAW_DIR = os.path.join(WORKING_DIR, 'raw_video')
OUTPUT_VIDEO = os.path.join(WORKING_DIR, 'final_youtube_short.mp4')
VOICEOVER_FILE = os.path.join(WORKING_DIR, 'ai_voiceover.wav')

os.makedirs(RAW_DIR, exist_ok=True)

# CELL 3: DOWNLOAD REEL (Direct CDN + Fallback)
print('📥 Downloading video...')
headers = {
    'User-Agent': 'Mozilla/5.0 Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json',
    'X-IG-App-ID': '936619743392459'
}

video_url = None
try:
    resp = requests.get(f'https://www.instagram.com/api/v1/media/{shortcode}/?__a=1&__d=dis', headers=headers, timeout=30)
    if resp.status_code == 200 and 'items' in resp.json():
        video_url = resp.json()['items'][0].get('video_versions', [{}])[0].get('url')
except: pass

if not video_url:
    print("⚠️ API fetch failed, trying instaloader fallback...")
    L = instaloader.Instaloader(download_videos=False, download_pictures=False)
    try:
        post = instaloader.Post.from_shortcode(L.context, shortcode)
        video_url = post.video_url
    except Exception as e:
        print(f"❌ All download methods failed: {e}")
        sys.exit(1)

v_resp = requests.get(video_url, stream=True, timeout=120)
v_resp.raise_for_status()
output_path = os.path.join(RAW_DIR, f'{username}_{shortcode}.mp4')
with open(output_path, 'wb') as f:
    for chunk in v_resp.iter_content(chunk_size=8192):
        if chunk: f.write(chunk)
print(f"✅ Downloaded: {os.path.basename(output_path)} ({os.path.getsize(output_path)//1024} KB)")

# CELL 4: GPU VOICEOVER (BARK)
print('🎙️ Generating AI voiceover on GPU T4...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
processor = AutoProcessor.from_pretrained('suno/bark-small')
model = BarkModel.from_pretrained('suno/bark-small').to(device)

inputs = processor(script_text, voice_preset='v2/en_speaker_6')
audio = model.generate(**{k: v.to(device) for k, v in inputs.items()})
audio_np = audio.cpu().numpy().squeeze()
wavfile.write(VOICEOVER_FILE, rate=24000, data=audio_np)
print(f'✅ Voiceover saved')
del model, processor; torch.cuda.empty_cache()

# CELL 5: FFMPEG RENDER (9:16 + Captions + Branding)
print('🎬 Rendering final video...')
hook_text = script_text.split('.')[0][:30] if script_text else "AI Tip"
ffmpeg_cmd = [
    'ffmpeg', '-y', '-i', output_path, '-i', VOICEOVER_FILE,
    '-filter_complex',
    f"[0:v]drawtext=text='{hook_text}':x=(w-text_w)/2:y=50:fontsize=28:fontcolor=white:box=1:boxcolor=black@0.7:enable='between(t,0,3)',drawtext=text='Save this!':x=(w-text_w)/2:y=h-70:fontsize=24:fontcolor=yellow:box=1:boxcolor=black@0.7:enable='between(t,10,14)',crop=ih*9/16:ih[v]",
    '-map', '[v]', '-map', '1:a', '-c:v', 'libx264', '-preset', 'fast', '-crf', '22',
    '-c:a', 'aac', '-b:a', '128k', '-shortest', OUTPUT_VIDEO
]
subprocess.run(ffmpeg_cmd, check=True, capture_output=True)
print(f'✅ Rendered: {OUTPUT_VIDEO} ({os.path.getsize(OUTPUT_VIDEO)//1024} KB)')

# CELL 6: UPLOAD TO YOUTUBE
print('📤 Uploading to YouTube...')
yt_url = None
upload_success = False
try:
    creds = Credentials(token=None, refresh_token=YT_REFRESH_TOKEN,
                        token_uri='https://oauth2.googleapis.com/token',
                        client_id=YT_CLIENT_ID, client_secret=YT_CLIENT_SECRET,
                        scopes=['https://www.googleapis.com/auth/youtube.upload'])
    if creds.expired: creds.refresh(Request())
    
    youtube = build('youtube', 'v3', credentials=creds)
    body = {
        'snippet': {'title': pipeline.get('youtube_title', 'AI Hack #shorts'),
                    'description': pipeline.get('youtube_description', ''),
                    'tags': pipeline.get('youtube_tags', ['AI', 'Shorts'])},
        'status': {'privacyStatus': 'public', 'selfDeclaredMadeForKids': False}
    }
    request = youtube.videos().insert(part=','.join(body.keys()), body=body, media_body=MediaFileUpload(OUTPUT_VIDEO, chunksize=-1, resumable=True))
    response = request.execute()
    yt_url = f"https://www.youtube.com/watch?v={response['id']}"
    upload_success = True
    print(f'🎉 YouTube: {yt_url}')
except Exception as e:
    print(f'⚠️ Upload failed: {e}')

# CELL 7: UPDATE GITHUB LEDGER
print('🔄 Updating GitHub ledger...')
try:
    led_url = f'https://api.github.com/repos/{GITHUB_USER}/{GITHUB_REPO}/contents/reel_queue.json'
    headers_gh = {'Authorization': f'token {GH_TOKEN}', 'Accept': 'application/vnd.github.v3+json'}
    resp_gh = requests.get(led_url, headers=headers_gh)
    current = json.loads(requests.utils.b64decode(resp_gh.json()['content']).decode())
    
    for entry in current.get('processed', []):
        if entry['url'] == reel_url and entry.get('status') == 'in_progress':
            entry['status'] = 'success' if upload_success else 'failed'
            if yt_url: entry['youtube_url'] = yt_url
            entry['completed_at'] = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
            break
            
    new_content = requests.utils.b64encode(json.dumps(current).encode()).decode()
    requests.put(led_url, headers=headers_gh, json={'message': 'Auto: Updated', 'content': new_content, 'sha': resp_gh.json()['sha']})
    print('✅ GitHub ledger updated')
except Exception as e:
    print(f'⚠️ Ledger warning: {e}')

print('\n🏆 PIPELINE COMPLETE!')